# 🎯 Sampling Design

**Description**: This notebook facilitates the __sampling design__ process for unbiased area estimation. It preprocesses the geospatial data and creates sample sets based on the specified allocation method and error targets.

In [ ]:
# Only change these if needed
CACHE_PATH = 'ignorefolder/cache'

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import time
import random
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import folium
from folium.plugins import MarkerCluster
import pandas as pd
import warnings
import datetime

from unbiased_area_estimation.region import Region
from unbiased_area_estimation.sampling_design import SamplingDesignPipeline
from unbiased_area_estimation.utils import get_unique_classes, get_nodata_value

warnings.simplefilter(action='ignore', category=FutureWarning)

# Cache path
os.makedirs(CACHE_PATH, exist_ok=True)

# Global state
step = 1
results = {}
sampling_design_pipeline = None
regions = None
class_merge_widgets = []
expected_accuracy_widgets = []
samples_per_class_widgets = []
expected_error_displays = []

# Output area for the step UI
out = widgets.Output()
# Create a dedicated output widget for progress monitoring
progress_out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '10px', 'margin': '10px 0', 'max-height': '300px', 'overflow': 'auto'})

# Create a status widget to show current step
status_widget = widgets.HTML(
    value="<div style='background-color: #f0f0f0; padding: 10px; border-radius: 5px;'><b>Current Step:</b> 1 - Set Output Folder and Sampling Method</div>",
    layout={'margin': '10px 0'}
)

# Display widgets in correct order
display(status_widget)
display(progress_out)
display(out)

reset_button = widgets.Button(description="Reset")

# --- Step 1 widgets ---
step_1_outputdir_header = widgets.HTML(value="<b>Set Outputfolder.</b><br/> Your sampling design and results will be saved here.")
output_dir = widgets.Text(description="Output Path:", value="/content/drive/MyDrive/unbiased_area_estimation", style={'description_width': 'initial'})
now = datetime.datetime.now()
now_str = now.strftime("%Y-%m-%d-%H-%M-%S")
run_name = widgets.Text(description="Run Name:", value=f"sample-design-{now_str}", style={'description_width': 'initial'})
step_1_header = widgets.HTML(value="<b>Set Sampling Method.</b>")
sampling_method = widgets.Dropdown(
    options=["Stratified", "Not implemented: Simple Random", "Not implemented: Two-Stage"],
    description="Sampling Method:",
    style={'description_width': 'initial'}
)
step1_button = widgets.Button(description="Next")
loading1 = widgets.Label("")

# --- Step 2 widgets ---
step_2_header = widgets.HTML(value="<b>Set Inputs.</b>")
map_path = widgets.Text(description="Map Path:", style={'description_width': 'initial'})
masks_path = widgets.Textarea(description="Mask Paths:", style={'description_width': 'initial'})
target_spatial_ref = widgets.Text(
    description="Target Spatial Ref",
    value="+proj=aea +lat_1=40 +lat_2=50 +lat_0=45 +lon_0=30 +x_0=0 +y_0=0 +ellps=WGS84 +datum=WGS84 +units=m +no_def",
    style={'description_width': 'initial'}
)
step2_button = widgets.Button(description="Next")
loading2 = widgets.Label("")

# --- Step 3 widgets ---
step_3_header = widgets.HTML(value="<b>Define Classes to Merge.</b>")
step3_button = widgets.Button(description="Next")
loading3 = widgets.Label("")

# --- Step 4 widgets ---
step4_header = widgets.HTML(value="<b>Set Sampling Parameters.</b>")
expected_accuracy_header = widgets.HTML(value="Expected Users Accuracies (per class)")
target_error = widgets.FloatText(description="Target Error:", value=0.01, style={'description_width': 'initial'})
allocation_method = widgets.Dropdown(
    options=["Proportional", "Neyman"],
    description="Allocation Method:",
    style={'description_width': 'initial'}
)
allocate_samples_header = widgets.HTML(value="<b>Allocate Samples</b>")
allocate_button = widgets.Button(description="Allocate")
expected_errors_header = widgets.HTML(value="<b>Expected Errors</b>")
expected_errors_header.layout.visibility = "hidden"
expected_error_button = widgets.Button(description="Update Expected Error", style={'description_width': 'initial'})
sampling_button = widgets.Button(description="Run Sampling", style={'description_width': 'initial'})
loading4 = widgets.Label("")

# Reset button
eset_button = widgets.Button(description="Reset")

# --- Helper functions ---
def log_progress(message, data=None):
    """Log progress messages and data to the progress output widget"""
    with progress_out:
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {message}")
        if data is not None:
            if isinstance(data, (list, dict, set)):
                print(f"Data: {data}")
            else:
                print(f"Data value: {data}")
            print("-" * 50)

def set_class_merge_widgets(classes):
    global class_merge_widgets, results
    class_merge_widgets.clear()
    class_merge_widgets.append(widgets.HTML(value="If you want to merge classes, please define the mapping below. For example, if you want to merge class 1 and 2, you can define 1:1 and 2:1."))
    nodata_val = get_nodata_value(results["map_path"])
    log_progress(f"Found nodata value", nodata_val)
    class_merge_widgets.append(widgets.HTML(value=f"Nodata value: {nodata_val}"))
    for class_id in classes:
        class_merge_widgets.append(widgets.IntText(description=f"{class_id}:", value=class_id))


def get_class_merge_dict():
    class_merge_dict = {}
    for widget in class_merge_widgets:
        if isinstance(widget, widgets.IntText):
            cid = widget.description.replace(':','')
            class_merge_dict[int(cid)] = widget.value
    log_progress("Class merge dictionary", class_merge_dict)
    return class_merge_dict


def get_expected_uas():
    expected_uas = {}
    for w in expected_accuracy_widgets:
        cid = w.description.replace(':','')
        expected_uas[int(cid)] = w.value
    log_progress("Expected user accuracies", expected_uas)
    return expected_uas


def get_updated_sampling_designs():
    updated = {}
    current_region = None
    for w in samples_per_class_widgets:
        if isinstance(w, widgets.Label):
            current_region = w.value
            updated[current_region] = {}
        elif isinstance(w, widgets.IntText):
            cid = w.description.replace(':','')
            updated[current_region][int(cid)] = w.value
    log_progress("Updated sampling designs", updated)
    return updated

# --- Step callbacks ---

def step1_submit(b):
    global step, results
    try:
        if step == 1:
            log_progress(f"Button clicked - Step 1 Next")
            results["sampling_method"] = sampling_method.value
            results["output_dir"] = os.path.join(output_dir.value, run_name.value)
            log_progress(f"Step 1 - Selected sampling method: {results['sampling_method']}")
            log_progress(f"Step 1 - Output directory: {results['output_dir']}")
            if os.path.exists(results["output_dir"]):
                log_progress(f"ERROR: Output folder {results['output_dir']} already exists!")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error: Output folder {results['output_dir']} already exists. Please choose another folder.</div>"))
                    display(step_1_outputdir_header, output_dir, run_name,
                            step_1_header, sampling_method, step1_button, loading1, reset_button)
                return
            
            # Make sure the directory exists
            try:
                os.makedirs(results["output_dir"])
                log_progress(f"Created directory: {results['output_dir']}")
            except Exception as dir_err:
                log_progress(f"ERROR creating directory: {str(dir_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error creating directory: {str(dir_err)}</div>"))
                    display(step_1_outputdir_header, output_dir, run_name,
                            step_1_header, sampling_method, step1_button, loading1, reset_button)
                return
                
            log_progress("Moving to step 2...")
            step2_ui()
            step += 1
            log_progress(f"Current step is now: {step}")
    except Exception as e:
        log_progress(f"ERROR in step1_submit: {str(e)}")
        with out:
            clear_output(wait=True)
            display(HTML(f"<div style='color: red; font-weight: bold;'>Error: {str(e)}</div>"))
            display(step_1_outputdir_header, output_dir, run_name,
                    step_1_header, sampling_method, step1_button, loading1, reset_button)


def step2_submit(b):
    global step, results, sampling_design_pipeline
    try:
        if step == 2:
            log_progress(f"Button clicked - Step 2 Next")
            
            # Validate inputs
            if not map_path.value:
                log_progress("ERROR: Map path is empty")
                with out:
                    clear_output(wait=True)
                    display(HTML("<div style='color: red; font-weight: bold;'>Error: Map path cannot be empty</div>"))
                    display(step_2_header, map_path, masks_path,
                            target_spatial_ref, step2_button, loading2, reset_button)
                return
                
            if not masks_path.value:
                log_progress("ERROR: Mask paths are empty")
                with out:
                    clear_output(wait=True)
                    display(HTML("<div style='color: red; font-weight: bold;'>Error: Mask paths cannot be empty</div>"))
                    display(step_2_header, map_path, masks_path,
                            target_spatial_ref, step2_button, loading2, reset_button)
                return
            
            results["map_path"] = map_path.value
            results["mask_paths"] = {Path(m).stem: m for m in masks_path.value.split("\n") if m}
            results["target_spatial_ref"] = target_spatial_ref.value
            
            log_progress(f"Step 2 - Map path: {results['map_path']}")
            log_progress(f"Step 2 - Mask paths: {results['mask_paths']}")
            
            # Check that the files actually exist
            if not os.path.exists(results["map_path"]):
                log_progress(f"ERROR: Map file does not exist: {results['map_path']}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error: Map file does not exist: {results['map_path']}</div>"))
                    display(step_2_header, map_path, masks_path,
                            target_spatial_ref, step2_button, loading2, reset_button)
                return
            
            for name, path in results["mask_paths"].items():
                if not os.path.exists(path):
                    log_progress(f"ERROR: Mask file does not exist: {path}")
                    with out:
                        clear_output(wait=True)
                        display(HTML(f"<div style='color: red; font-weight: bold;'>Error: Mask file does not exist: {path}</div>"))
                        display(step_2_header, map_path, masks_path,
                                target_spatial_ref, step2_button, loading2, reset_button)
                    return
            
            try:
                sampling_design_pipeline = SamplingDesignPipeline(
                    sampling_method=results["sampling_method"],
                    output_path=results["output_dir"],
                    use_cached=True,
                    cache_path=CACHE_PATH
                )
                log_progress("Created SamplingDesignPipeline successfully")
            except Exception as pipeline_err:
                log_progress(f"ERROR creating pipeline: {str(pipeline_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error creating pipeline: {str(pipeline_err)}</div>"))
                    display(step_2_header, map_path, masks_path,
                            target_spatial_ref, step2_button, loading2, reset_button)
                return
                
            loading2.value = "Identifying all classes..."
            log_progress("Identifying all classes in map...")
            
            try:
                results["classes"] = get_unique_classes(map_path=results["map_path"])
                log_progress("Found classes:", results["classes"])       
                set_class_merge_widgets(results["classes"])
            except Exception as classes_err:
                log_progress(f"ERROR identifying classes: {str(classes_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error identifying classes: {str(classes_err)}</div>"))
                    display(step_2_header, map_path, masks_path,
                            target_spatial_ref, step2_button, loading2, reset_button)
                loading2.value = ""
                return
                
            loading2.value = ""
            log_progress("Moving to step 3...")
            step3_ui()
            step += 1
            log_progress(f"Current step is now: {step}")
    except Exception as e:
        log_progress(f"ERROR in step2_submit: {str(e)}")
        with out:
            clear_output(wait=True)
            display(HTML(f"<div style='color: red; font-weight: bold;'>Error: {str(e)}</div>"))
            display(step_2_header, map_path, masks_path,
                    target_spatial_ref, step2_button, loading2, reset_button)


def step3_submit(b):
    global step, results, sampling_design_pipeline, regions
    try:
        if step == 3:
            log_progress(f"Button clicked - Step 3 Next")
            loading3.value = "Preprocessing files. This might take a while..."
            
            try:
                results["class_merge_dict"] = get_class_merge_dict()
                results["classes"] = set(results["class_merge_dict"].values())
                log_progress(f"Step 3 - Classes after merging: {results['classes']}")
            except Exception as merge_err:
                log_progress(f"ERROR getting class merge dict: {str(merge_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error getting class merge dictionary: {str(merge_err)}</div>"))
                    display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)
                loading3.value = ""
                return
                
            log_progress("Preprocessing files...")
            try:
                regions = sampling_design_pipeline.preprocess(
                    map_path=results["map_path"],
                    mask_paths=results["mask_paths"],
                    target_spatial_ref=results["target_spatial_ref"],
                    class_merge_map=results["class_merge_dict"]
                )
                if not regions:
                    log_progress("WARNING: No regions found during preprocessing")
                    with out:
                        clear_output(wait=True)
                        display(HTML("<div style='color: orange; font-weight: bold;'>Warning: No regions found during preprocessing</div>"))
                        display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)
                    loading3.value = ""
                    return
                    
                log_progress(f"Preprocessing complete. Regions found: {[r.name for r in regions]}")
            except Exception as preprocess_err:
                log_progress(f"ERROR during preprocessing: {str(preprocess_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error during preprocessing: {str(preprocess_err)}</div>"))
                    display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)
                loading3.value = ""
                return
                
            loading3.value = "Pixel counting..."
            log_progress("Counting pixels by class...")
            try:
                results["pixel_counts"] = []
                for r in regions:
                    try:
                        pc = r.get_pixel_counts_by_class()
                        results["pixel_counts"].append(pc)
                        log_progress(f"Pixel counts for region {r.name}:", pc)
                    except Exception as pc_err:
                        log_progress(f"ERROR counting pixels for region {r.name}: {str(pc_err)}")
                        # Continue with other regions
                
                log_progress("Calculating areas...")
                results["areas"] = []
                for r in regions:
                    try:
                        area = r.get_areas()
                        results["areas"].append(area)
                        log_progress(f"Areas for region {r.name}:", area)
                    except Exception as area_err:
                        log_progress(f"ERROR calculating areas for region {r.name}: {str(area_err)}")
                        # Continue with other regions
            except Exception as count_err:
                log_progress(f"ERROR during pixel counting/area calculation: {str(count_err)}")
                with out:
                    clear_output(wait=True)
                    display(HTML(f"<div style='color: red; font-weight: bold;'>Error during pixel counting/area calculation: {str(count_err)}</div>"))
                    display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)
                loading3.value = ""
                return
                
            loading3.value = ""
            # Clear previous widgets if any
            expected_accuracy_widgets.clear()
            for cid in results["classes"]:
                expected_accuracy_widgets.append(widgets.FloatText(description=f"{cid}:", value=0.85))
                
            log_progress("Moving to step 4...")
            step4_ui()
            step += 1
            log_progress(f"Current step is now: {step}")
    except Exception as e:
        log_progress(f"ERROR in step3_submit: {str(e)}")
        with out:
            clear_output(wait=True)
            display(HTML(f"<div style='color: red; font-weight: bold;'>Error: {str(e)}</div>"))
            display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)
        loading3.value = ""


def allocate_samples(b):
    global regions, sampling_design_pipeline, samples_per_class_widgets, expected_error_displays
    loading4.value = "Computing num samples and allocating..."
    log_progress("Computing samples allocation...")
    
    expected_uas = get_expected_uas()
    t_error = target_error.value
    alloc_method = allocation_method.value
    
    log_progress(f"Target error: {t_error}, Allocation method: {alloc_method}")
    
    designs, details = sampling_design_pipeline.create_and_save_sampling_designs(
        regions=regions,
        expected_uas=expected_uas,
        target_error=t_error,
        allocation_method_name=alloc_method
    )
    
    log_progress("Sampling design results:", designs)
    
    # build detail widgets
    detail_widgets = []
    for region, sd in details.items():
        acc = widgets.Accordion(children=[widgets.HTML(sd.to_html())])
        acc.set_title(0, f"{region} - Sampling Design Details")
        acc.selected_index = None
        detail_widgets.append(acc)
    results["sampling_designs"] = designs
    results["sampling_design_details"] = details

    samples_per_class_widgets.clear()
    expected_error_displays.clear()
    for region_name, sd in designs.items():
        samples_per_class_widgets.append(widgets.Label(value=region_name))
        for cid, val in sd.items():
            samples_per_class_widgets.append(widgets.IntText(description=f"{cid}:", value=val))
        expected_error_displays.append(widgets.Label(value=f"{region_name}: -"))

    expected_errors_header.layout.visibility = "visible"
    update_expected_error()

    with out:
        out.clear_output(wait=True)
        display(step4_header, expected_accuracy_header, *expected_accuracy_widgets,
                target_error, allocate_samples_header, allocation_method,
                allocate_button, *samples_per_class_widgets,
                *detail_widgets, expected_errors_header,
                expected_error_button, *expected_error_displays,
                sampling_button, loading4, reset_button)
    loading4.value = ""


def update_expected_error():
    loading4.value = "Computing new expected errors..."
    log_progress("Computing expected target errors...")
    
    updated_designs = get_updated_sampling_designs()
    region_names = [r.name for r in regions]
    
    errs = sampling_design_pipeline.get_expected_target_errors(
        region_names=region_names,
        sampling_designs=updated_designs
    )
    
    log_progress("Expected errors by region:", errs)
    
    for lbl in expected_error_displays:
        name = lbl.value.split(":")[0]
        lbl.value = f"{name}: {float(errs[name]):.4f}"
    loading4.value = ""


def run_sampling(b):
    global regions, sampling_design_pipeline
    loading4.value = "Sampling..."
    log_progress("Running sampling...")
    
    updated = get_updated_sampling_designs()
    sample_sets = sampling_design_pipeline.sample_and_save(
        regions=regions, sampling_designs=updated
    )
    
    log_progress(f"Sampling completed! Sample counts by region: {[len(df) for df in sample_sets.values()]}")
    loading4.value = "Sampling completed!"

    all_samples = pd.concat(list(sample_sets.values()))
    log_progress(f"Total samples: {len(all_samples)}")
    
    def rand_color(): return "#{:06x}".format(random.randint(0, 0xFFFFFF))
    classes = all_samples["stratum_id"].unique()
    cmap = {cid: rand_color() for cid in classes}

    gdfs = [gpd.read_file(fp).to_crs(epsg=4326) for fp in results["mask_paths"].values()]
    merged = gpd.GeoDataFrame(geometry=[gdf.geometry.unary_union for gdf in gdfs], crs="EPSG:3857")
    center = merged.geometry.centroid.iloc[0]
    m = folium.Map(location=[center.y, center.x], zoom_start=4, tiles="cartodbpositron")
    cluster = MarkerCluster().add_to(m)
    for _, row in all_samples.iterrows():
        folium.CircleMarker(
            location=[row["LAT"], row["LON"]],
            radius=6,
            color=cmap[row["stratum_id"]],
            fill=True,
            fill_color=cmap[row["stratum_id"]],
            fill_opacity=0.7,
            popup=f'Stratum ID: {row["stratum_id"]}'
        ).add_to(cluster)
    legend = '<div style="position: fixed; bottom: 50px; left: 50px; width: 200px; background-color: white; z-index:9999; font-size:14px; padding:10px; border-radius:5px;">\n<b>Legend</b><br>'
    for cid, col in cmap.items(): legend += f'<i class="fa fa-circle" style="color:{col}"></i> Stratum {cid}<br>'
    legend += '</div>'
    m.get_root().html.add_child(folium.Element(legend))

    with out:
        display(m)
        display(HTML("<div style='border:1px solid #ccc; padding:10px; margin-top:10px; background:#f9f9f9;'><b>Action Required:</b> Please review the sampled points in a tool like QGIS. Load the original raster and set symbology to unique values. The load the CSV by going to Add Layer -> Add Delimiter Layer and set the CRS to EPSG:4326 in the loading options.</div>"))


def reset_all(b):
    global step, results, sampling_design_pipeline, regions
    step = 1
    results = {}
    sampling_design_pipeline = None
    regions = None
    class_merge_widgets.clear()
    expected_accuracy_widgets.clear()
    samples_per_class_widgets.clear()
    expected_error_displays.clear()
    loading1.value = loading2.value = loading4.value = ""
    log_progress("Reset application state")
    
    # Clear progress output
    with progress_out:
        progress_out.clear_output()
        print("Progress log cleared")
    
    step1_ui()

# --- Add debug callback to track button clicks ---
def debug_button_click(b):
    log_progress(f"Button '{b.description}' was clicked")
    
# --- Attach callbacks ---
def safe_handler(handler):
    """Wrap a button handler to catch any uncaught exceptions"""
    def safe_function(b):
        try:
            log_progress(f"Button '{b.description}' clicked - calling handler")
            handler(b)
        except Exception as e:
            log_progress(f"CRITICAL ERROR in button handler: {str(e)}")
            import traceback
            log_progress(f"Traceback: {traceback.format_exc()}")
            with out:
                clear_output(wait=True)
                display(HTML(f"<div style='color: red; font-weight: bold;'>Critical Error: {str(e)}</div>"))
                display(HTML(f"<div style='color: red; font-family: monospace; white-space: pre-wrap;'>{traceback.format_exc()}</div>"))
                display(reset_button)
    return safe_function

# Safe handlers for all buttons
step1_button.on_click(safe_handler(step1_submit))
step2_button.on_click(safe_handler(step2_submit))
step3_button.on_click(safe_handler(step3_submit))
allocate_button.on_click(safe_handler(allocate_samples))
expected_error_button.on_click(safe_handler(lambda b: update_expected_error()))
sampling_button.on_click(safe_handler(run_sampling))
reset_button.on_click(safe_handler(reset_all))

# --- Step UI functions ---
def update_status(step_num, description):
    """Update the status widget to show current step"""
    status_widget.value = f"<div style='background-color: #f0f0f0; padding: 10px; border-radius: 5px;'><b>Current Step:</b> {step_num} - {description}</div>"
    log_progress(f"Updated status to Step {step_num} - {description}")

def step1_ui():
    update_status(1, "Set Output Folder and Sampling Method")
    with out:
        out.clear_output(wait=True)
        display(step_1_outputdir_header, output_dir, run_name,
                step_1_header, sampling_method, step1_button, loading1, reset_button)

def step2_ui():
    update_status(2, "Set Inputs")
    with out:
        out.clear_output(wait=True)
        display(step_2_header, map_path, masks_path,
                target_spatial_ref, step2_button, loading2, reset_button)

def step3_ui():
    update_status(3, "Define Classes to Merge")
    with out:
        out.clear_output(wait=True)
        display(step_3_header, *class_merge_widgets, step3_button, loading3, reset_button)

def step4_ui():
    update_status(4, "Set Sampling Parameters")
    with out:
        out.clear_output(wait=True)
        display(step4_header, expected_accuracy_header, *expected_accuracy_widgets,
                target_error, allocate_samples_header, allocation_method,
                allocate_button, loading4, reset_button)

# Initialize UI
log_progress("Application initialized")
step1_ui()

HTML(value="<div style='background-color: #f0f0f0; padding: 10px; border-radius: 5px;'><b>Current Step:</b> 1 …

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

Output()